# Customer Churn Prediction — Full Corrected Pipeline

This notebook fixes the two issues found in the original version:

1. **Submission file mismatch** — the old code wrote `submission.csv` with no ID column and a
   column name (`Prediction`) that almost certainly didn't match what Kaggle expected. The model
   itself scored ~0.80 F1 in cross-validation, but the leaderboard score was 0.73 because the
   *file*, not the model, was wrong.
2. **Lost test IDs** — the test set's ID column was never preserved through the cleaning /
   encoding pipeline, so there was nothing correct to attach to predictions even if the column
   name had been right.

**⚠️ Action required from you:** in the cell marked `# >>> CONFIRM THIS <<<` below, replace
`"customer_id"` with whatever your actual ID column is called, and check Kaggle's
`sample_submission.csv` for the exact column names/order expected. Everything else is ready to run.


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', 100)
sns.set_style('whitegrid')


## 2. Load data

We immediately save the test set's ID column to a separate variable, **before any cleaning
touches `test`**. This is the single most important fix — every later step can drop, reorder,
or filter `test` and we will still be able to rebuild a correctly-aligned submission at the end.

In [ ]:
train = pd.read_csv("training_dataset.csv")
test = pd.read_csv("testing_dataset.csv")

# Strip whitespace from column names immediately
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


In [ ]:
# >>> CONFIRM THIS <<<
# Replace 'customer_id' with whatever the real ID column is called in your test set.
# Check test.columns below to find it, and check Kaggle's sample_submission.csv for
# the exact column name(s) your submission file needs to use.
print(test.columns.tolist())

ID_COL = "customer_id"   # <-- change if needed

test_ids = test[ID_COL].copy()   # saved untouched, will be used for the final submission


## 3. Exploratory Data Analysis (EDA)

Goal: understand the target distribution, spot data quality issues (missing values, outliers,
impossible values like negative usage), and see which features look related to churn — all
*before* we engineer features or model anything.

### 3.1 Basic structure

In [ ]:
train.info()


In [ ]:
train.describe(include='all').T


### 3.2 Missing values

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing)

if len(missing) > 0:
    plt.figure(figsize=(8, 4))
    missing.plot(kind='bar')
    plt.title("Missing values per column (train)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values in train.")


### 3.3 Duplicate rows

In [ ]:
dupes = train.duplicated().sum()
print(f"Duplicate rows in train: {dupes}")


### 3.4 Target distribution (`churn`)

This matters a lot: if churn is imbalanced (which telecom churn almost always is), accuracy is
a misleading metric and F1 / recall on the minority class matter more. It also tells us whether
techniques like SMOTE are worth using.

In [ ]:
churn_counts = train['churn'].value_counts()
churn_pct = train['churn'].value_counts(normalize=True) * 100

print(churn_counts)
print()
print(churn_pct.round(2))

plt.figure(figsize=(5, 4))
sns.countplot(x='churn', data=train)
plt.title("Churn distribution")
plt.xlabel("Churn (0 = stayed, 1 = churned)")
plt.show()


### 3.5 Numeric feature distributions and outliers

In [ ]:
num_cols_eda = train.select_dtypes(include='number').drop(columns=['churn'], errors='ignore').columns

fig, axes = plt.subplots(nrows=(len(num_cols_eda) + 2) // 3, ncols=3, figsize=(15, 4 * ((len(num_cols_eda) + 2) // 3)))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    sns.histplot(train[col], bins=40, ax=axes[i], kde=True)
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
# Sanity check for impossible values (e.g. negative calls/sms/data, which shouldn't exist)
for col in num_cols_eda:
    n_negative = (train[col] < 0).sum()
    if n_negative > 0:
        print(f"{col}: {n_negative} negative values (min={train[col].min()})")


### 3.6 Numeric features vs churn

In [ ]:
fig, axes = plt.subplots(nrows=(len(num_cols_eda) + 2) // 3, ncols=3, figsize=(15, 4 * ((len(num_cols_eda) + 2) // 3)))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    sns.boxplot(x='churn', y=col, data=train, ax=axes[i])
    axes[i].set_title(f"{col} by churn")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


### 3.7 Categorical features vs churn

In [ ]:
cat_cols_eda = train.select_dtypes(exclude='number').columns.tolist()
print("Categorical columns:", cat_cols_eda)

for col in cat_cols_eda:
    if train[col].nunique() <= 15:  # skip high-cardinality columns like exact dates
        plt.figure(figsize=(6, 4))
        churn_rate = train.groupby(col)['churn'].mean().sort_values(ascending=False)
        churn_rate.plot(kind='bar')
        plt.title(f"Churn rate by {col}")
        plt.ylabel("Churn rate")
        plt.tight_layout()
        plt.show()


### 3.8 Correlation between numeric features

In [ ]:
plt.figure(figsize=(10, 8))
corr = train[num_cols_eda.tolist() + ['churn']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()


**EDA takeaways to act on:**
- Note the churn class balance from 3.4 — if it's skewed (e.g. ~80/20), keep using F1 as the
  scoring metric, not accuracy, and consider SMOTE (we do, below) or `class_weight='balanced'`.
- Note any negative values found in 3.5 — these are most likely encoding errors (e.g. -1 used as
  a missing-value placeholder) and should be treated as missing, not real negative usage.
- Note which categorical/numeric features show a visibly different churn rate in 3.6/3.7 — these
  are good candidates to keep; near-flat ones are weaker signal.


## 4. Cleaning

We deduplicate, fix impossible negative values (treat as missing rather than silently keeping
them), and handle infinities — all consistently on both `train` and `test`.

In [ ]:
train = train.drop_duplicates().reset_index(drop=True)

# Treat impossible negative values in count/usage columns as missing
usage_like_cols = ['calls_made', 'sms_sent', 'data_used']
for col in usage_like_cols:
    if col in train.columns:
        train.loc[train[col] < 0, col] = np.nan
    if col in test.columns:
        test.loc[test[col] < 0, col] = np.nan

train = train.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

print("Train shape after cleaning:", train.shape)
print("Test shape after cleaning:", test.shape)


## 5. Feature engineering

Done once, identically on `train` and `test`, using `.replace(0, np.nan)` on denominators so we
get a clean missing value instead of an `inf` from dividing by zero (the original notebook used
`.replace(0, 1)`, which silently distorts the ratio instead of marking it missing — small but
worth fixing).

In [ ]:
for df in (train, test):
    safe_tenure = df['customer_tenure_days'].replace(0, np.nan)
    safe_data_used = df['data_used'].replace(0, np.nan)

    df['calls_per_day'] = df['calls_made'] / safe_tenure
    df['sms_per_day'] = df['sms_sent'] / safe_tenure
    df['data_per_day'] = df['data_used'] / safe_tenure
    df['salary_data_ratio'] = df['estimated_salary'] / safe_data_used
    df['activity_score'] = df['calls_per_day'] + df['sms_per_day'] + df['data_per_day']
    df['total_usage'] = df['calls_made'] + df['sms_sent'] + df['data_used']

train.replace([np.inf, -np.inf], np.nan, inplace=True)
test.replace([np.inf, -np.inf], np.nan, inplace=True)

print(train.filter(like='_per_day').describe().T)


## 6. Split features/target and impute

`num_cols` / `cat_cols` are computed from `X_train` (features only, no `churn`) so the target
never accidentally gets treated as a feature to impute or encode — this was a small bug in the
original version. Imputers are fit on train only and applied (`.transform`, not `.fit_transform`)
to test, which avoids leaking test statistics into training.

In [ ]:
X_train = train.drop(columns=['churn'])
y_train = train['churn']
X_test = test.copy()

num_cols = X_train.select_dtypes(include='number').columns
cat_cols = X_train.select_dtypes(exclude='number').columns

print("Numeric columns:", list(num_cols))
print("Categorical columns:", list(cat_cols))


In [ ]:
num_imputer = SimpleImputer(strategy='median')
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
    X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

print("Remaining missing in X_train:", X_train.isnull().sum().sum())
print("Remaining missing in X_test:", X_test.isnull().sum().sum())


## 7. Encode categoricals

One-hot encode, then align `X_test` to `X_train`'s columns. This keeps the ID column issue
separate entirely — `test_ids` was saved in section 2 and never enters `X_train`/`X_test`, so
it can't be dropped or mismatched here.

In [ ]:
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
assert list(X_train.columns) == list(X_test.columns), "Column mismatch between train and test!"


## 8. Handle class imbalance with SMOTE

Applied only to the training data (never to test/validation), which is correct — SMOTE should
never see or influence anything used for evaluation.

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(y_train_smote.value_counts())


## 9. Feature importance check (optional, informational)

Just for insight into which engineered features matter — not used to drop columns here, since
the original 0.001 threshold barely removed anything useful anyway.

In [ ]:
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_train, y_train)

feature_importance = pd.Series(
    rf_selector.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

print(feature_importance.head(20))

plt.figure(figsize=(8, 6))
feature_importance.head(20).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title("Top 20 feature importances")
plt.tight_layout()
plt.show()


## 10. Train model and validate with cross-validation

Cross-validation is run on the SMOTE-resampled training data, mirroring the original approach,
so this number is comparable to what you saw before (~0.80 F1). The key difference is everything
downstream of this is now actually correct.

In [ ]:
best_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    best_model, X_train_smote, y_train_smote, cv=cv, scoring='f1', n_jobs=-1
)

print("F1 scores per fold:", scores)
print("Average F1 score:", scores.mean())


In [ ]:
best_model.fit(X_train_smote, y_train_smote)


## 11. Predict on test set

In [ ]:
predictions = best_model.predict(X_test)

print("Prediction counts:")
print(pd.Series(predictions).value_counts())
print("\nFirst 20 predictions:", predictions[:20])


## 12. Build the submission file correctly

This is the fix that matters most. `test_ids` was saved untouched in section 2, so it lines up
row-for-row with `predictions` (both derived from the same original `test` row order, and nothing
in between ever sorted, filtered, or shuffled either one).

**Before running this cell:** open Kaggle's `sample_submission.csv` and make sure the column
names below (`ID_COL` and `"churn"`) exactly match it — including capitalization.

In [ ]:
submission = pd.DataFrame({
    ID_COL: test_ids,
    "churn": predictions   # <-- rename this key if Kaggle's sample_submission uses a different name
})

print(submission.head())
print(submission.shape)

submission.to_csv("submission.csv", index=False)
print("Submission file saved successfully")
